In [1]:
import pandas as pd
import numpy as np

test = pd.read_csv("data/clean/test_scored.csv")
scorecard = pd.read_csv("outputs/scorecard_points.csv")
cutoff = 548

inf = np.inf
bins = {
    "utilisation":       [-inf, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, inf],
    "age":               [-inf, 30, 40, 50, 60, 70, inf],
    "late_30_59":        [-inf, 0, 1, 2, inf],
    "late_60_89":        [-inf, 0, 1, inf],
    "late_90":           [-inf, 0, 1, inf],
    "debt_ratio":        [-inf, 0.2, 0.4, 0.6, 0.8, 1.0, inf],
    "monthly_income":    [-inf, 2000, 3500, 5000, 7500, 10000, inf],
    "open_credit_lines": [-inf, 2, 5, 10, 15, inf],
    "real_estate_loans": [-inf, 0, 1, 2, inf],
    "dependents":        [-inf, 0, 1, 2, inf],
}
features = list(bins.keys())

def make_bins(data, col):
    b = pd.cut(data[col], bins[col])
    return b.cat.add_categories("Missing").fillna("Missing")

In [2]:
max_pts = scorecard.groupby("feature")["points"].max()

lost = pd.DataFrame(index=test.index)
for col in features:
    pts = scorecard[scorecard["feature"] == col].set_index("bin")["points"]
    lost[col] = max_pts[col] - make_bins(test, col).astype(str).map(pts)

lost.head()

,utilisation,age,late_30_59,late_60_89,late_90,debt_ratio,monthly_income,open_credit_lines,real_estate_loans,dependents
0,0,13,0,23,36,2,0,0,9,4
1,0,0,0,0,0,2,1,0,0,4
2,40,4,20,0,0,2,1,0,9,4
3,0,0,20,0,0,0,0,0,0,4
4,0,10,0,0,0,1,0,0,2,4


In [3]:
reason_text = {
    "utilisation": "High use of available credit limits",
    "late_30_59": "Recent missed payments (30–59 days late)",
    "late_60_89": "Missed payments (60–89 days late)",
    "late_90": "Serious arrears (90+ days late)",
    "debt_ratio": "High debt repayments compared with income",
    "monthly_income": "Income level, or income not provided",
    "age": "Age band (flagged for fairness review)",
    "open_credit_lines": "Number of open credit accounts",
    "real_estate_loans": "Number of property-related loans",
    "dependents": "Number of dependants",
}

declined = test[test["score"] < cutoff].copy()
top2 = lost.loc[declined.index].apply(lambda r: r.nlargest(2).index.tolist(), axis=1)

declined["reason_1"] = top2.str[0].map(reason_text)
declined["reason_2"] = top2.str[1].map(reason_text)
declined["points_short"] = cutoff - declined["score"]

print("Declined applicants:", len(declined))
declined[["score", "points_short", "reason_1", "reason_2"]].head(10)

Declined applicants: 5827


,score,points_short,reason_1,reason_2
0,541.0,7.0,Serious arrears (90+ days late),Missed payments (60–89 days late)
10,424.0,124.0,High use of available credit limits,Recent missed payments (30–59 days late)
17,545.0,3.0,High use of available credit limits,High debt repayments compared with income
19,530.0,18.0,High use of available credit limits,Recent missed payments (30–59 days late)
38,529.0,19.0,High use of available credit limits,Recent missed payments (30–59 days late)
40,512.0,36.0,High use of available credit limits,Missed payments (60–89 days late)
41,539.0,9.0,High use of available credit limits,Missed payments (60–89 days late)
50,515.0,33.0,Serious arrears (90+ days late),High use of available credit limits
53,427.0,121.0,Serious arrears (90+ days late),Recent missed payments (30–59 days late)
65,524.0,24.0,High use of available credit limits,Recent missed payments (30–59 days late)


In [4]:
summary = (declined["reason_1"].value_counts(normalize=True) * 100).round(1)
summary

reason_1
High use of available credit limits         73.3
Serious arrears (90+ days late)             18.7
Recent missed payments (30–59 days late)     6.2
Missed payments (60–89 days late)            1.9
Name: proportion, dtype: float64

In [5]:
declined[["score", "points_short", "reason_1", "reason_2", "default_2yrs"]].to_csv(
    "outputs/decline_reasons.csv", index=False)

In [6]:
declined["band"] = pd.cut(declined["points_short"], [0, 10, 30, 1000], labels=["Within 10 pts", "10–30 pts", "30+ pts"])
declined.groupby("band", observed=True).agg(
    applicants=("score", "size"), bad_rate=("default_2yrs", "mean")).assign(bad_rate=lambda d: (d["bad_rate"] * 100).round(1))

,applicants,bad_rate
band,,
Within 10 pts,1406,14.8
10–30 pts,1933,23.4
30+ pts,2488,47.1
